# Demo: Memory and Perplexity: Udacity Solution

In [ ]:
import math
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

### Helper functions

In [ ]:
# Short Prewritten Passage about AI as input text
def select_text():
    return \"\"\"
'Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural'
' intelligence displayed by humans and animals. Leading AI textbooks define the field as the study'
' of intelligent agents: any system that perceives its environment and takes actions that maximize'
' its chance of achieving its goals. Colloquially, the term \\"artificial intelligence\\" is often'
' used to describe machines that mimic cognitive functions that humans associate with the human mind,'
' such as learning and problem-solving.'
\"\"\"

# Helper function to load model and tokenizer
def load_model_and_tokenizer(model_name: str):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype
    )
    model.to(device)
    model.eval()
    return tokenizer, model, device

# Tokenize the text using the tokenizer from above
def tokenize_with_labels(tokenizer, text: str):
    inputs = tokenizer(text, return_tensors="pt")
    inputs["labels"] = inputs["input_ids"].clone()
    input_len = inputs["input_ids"].size(1)
    return inputs, input_len

### Core Functions : Compute Peak Memory, Compute Perplexity

1) For accurate GPU Profiling , you only need these
```
torch.cuda.reset_peak_memory_stats(device)
with torch.no_grad():
    outputs = model(...)
torch.cuda.synchronize()
peak = torch.cuda.max_memory_allocated(device)
```

You do not need the following
```
gc.collect()
torch.cuda.empty_cache()
```

2) You only profile GPU.
   - Typically GPU is the bottle neck and models are deployed on GPU
   - Pytorch's CPU profiling tools are not great and not very accurate
     
3) model: inputs
   - hugging face model typical inputs are input_ids (deploy them to the device)
   - attention masks: optional
   - labels:so the model computes the loss internally

4) model: outputs usually contains
   - loss: which we will use 
   - logits
   - other optional
     
5) see Readme.md for difference between model and model.generate()


In [3]:
def compute_peak_memory_and_loss(model, inputs, device):
    torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs.get("attention_mask", None).to(device) if inputs.get("attention_mask") is not None else None,
            labels=inputs["labels"].to(device)
        )
    if device.type == "cuda":
        torch.cuda.synchronize()
        peak_bytes = torch.cuda.max_memory_allocated(device)
        peak_mib = peak_bytes / 1024**2
    else:
        peak_mib = float("nan")
    return peak_mib, outputs.loss.item()

def compute_perplexity(loss: float):
    return math.exp(loss)

### Orchestrating Start Function

In [1]:
def start():
    # 1. Load
    tokenizer, model, device = load_model_and_tokenizer(MODEL_NAME)
    print(f"Using device: {device}")

    # 2. Select & tokenize
    text = select_text()
    inputs, input_len = tokenize_with_labels(tokenizer, text)
    print(f"Tokenized length: {input_len}")

    # 3. Measure peak memory & loss
    peak_mem, loss = compute_peak_memory_and_loss(model, inputs, device)
    print(f"Peak GPU memory: {peak_mem:.1f} MiB")

    # 4. Compute perplexity
    ppl = compute_perplexity(loss)
    print(f"Next-token perplexity: {ppl:.3f}")



In [ ]:
# This "TinyLlama/TinyLlama-1.1B-Chat-v1.0" is too big for a 4GB VRAM GPU like 3050. Will cause out of memory errors
# PREFERRED_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# This smaller model better on GEFORCE 3050
PREFERRED_MODEL = "facebook/opt-350m"
# PREFERRED_MODEL = "unsloth/LFM2-700M"

start()